In [1]:
import pandas as pd, numpy as np, os, sys, torch, torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
sys.path.insert(0, "..")  # ensure current directory is in the path
from base_splits import build_base_splits, load_splits, save_splits, summarize_splits, load_dataframe
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             confusion_matrix)
from models.lstm_xgb import LSTM_XGB
from models.cnn_lstm import CNN_LSTM_v1, CNN_LSTM_v2
from utils import reset_gpu_peak_memory, get_gpu_peak_memory_mb, measure_inference_time, Timer, count_parameters
scaler = StandardScaler()

splits = load_splits(path="../base_splits.pkl")  
print(summarize_splits(splits))

device = "cuda:1" if torch.cuda.is_available() else "cpu"
print("device:", torch.cuda.get_device_name(device) if "cuda" in device else device)

cols = splits[0].train.columns[1:-1]


Loaded base splits for 8 classes from /home/maddie/cd-study/Stages/base_splits.pkl
   label  train_n  val_n  test_n  time_start   time_end
0      0     1000    300     300    3.993639   4.153525
1      1     1000    300     300    9.156422   9.316307
2      2     1000    300     300    4.507887   4.667771
3      3     1000    300     300    4.866492   5.026376
4      4     1000    300     300    1.528859   1.688743
5      5     1000    300     300    8.582310   8.742194
6      6     1000    300     300    8.765543   8.925427
7      7     1000    300     300   10.680525  10.840409
device: NVIDIA A30


In [2]:
# Z-scale
dct = dict()
scaler.fit(splits[0].train[cols])
for i in range(len(splits)):
    dct[i] = dict()
    dct[i].update({
        # FIX: was splits[0].train[cols] for every i, which silently duplicated
        # class 0's training rows into every other class (only Fault differed).
        "train": pd.DataFrame(scaler.transform(splits[i].train[cols]), columns=cols, index=splits[i].train.index).assign(Fault=i),
        "val": pd.DataFrame(scaler.transform(splits[i].val[cols]), columns=cols, index=splits[i].val.index).assign(Fault=i),
        "test": pd.DataFrame(scaler.transform(splits[i].test[cols]), columns=cols, index=splits[i].test.index).assign(Fault=i),

    })


In [3]:
def to_tensors(df, cols):
    """Convert a (features + Fault) DataFrame into model-ready tensors.
    X: (N, 1, len(cols)) so the LSTM sees the len(cols) features as a
       length-len(cols) sequence with 1 channel each (matches LSTM_XGB's
       expected input shape).
    y: (N,) integer Fault labels.
    """
    X = torch.from_numpy(df[cols].to_numpy(dtype="float32")).unsqueeze(1)
    y = torch.from_numpy(df["Fault"].to_numpy(dtype="int64"))
    return X, y


def load_scenario(dct, cols):
    """Build combined 8-class train/val/test tensors directly from the
    in-memory `dct` dict (built in the Z-scale cell), instead of reading
    per-scenario CSVs off disk.

    dct is keyed by class label: dct[i]["train"/"val"/"test"] is a
    per-class DataFrame of z-scored features + a Fault column. We
    concatenate across classes to get the full multiclass split.
    """
    train_df = pd.concat([dct[i]["train"] for i in sorted(dct)], axis=0)
    val_df   = pd.concat([dct[i]["val"]   for i in sorted(dct)], axis=0)
    test_df  = pd.concat([dct[i]["test"]  for i in sorted(dct)], axis=0)
    return (to_tensors(train_df, cols),
            to_tensors(val_df,   cols),
            to_tensors(test_df,  cols))


def run_scenario(scenario_idx, dct, cols, device,
                 epochs=70, lr=1e-2, weight_decay=1e-4, seed=0, verbose=True):
    (X_tr, y_tr), (X_va, y_va), (X_te, y_te) = load_scenario(dct, cols)

    model = LSTM_XGB(n_classes=8, lstm_hidden=32, device=device, seed=seed)
    n_params, params_by_type = count_parameters(model.backbone)  # LSTM only
    reset_gpu_peak_memory(device)

    if verbose:
        print(f"\n=== Scenario {scenario_idx} (LSTM_XGB) ===")
        print(f"  LSTM params: {n_params:,}  breakdown: {params_by_type}")

    # ---- Stage 1: LSTM training ----
    with Timer(device) as stage1_timer:
        model.fit_lstm(X_tr, y_tr, X_val=X_va, y_val=y_va,
                       epochs=epochs, lr=lr, weight_decay=weight_decay,
                       batch_size=50, seed=seed, verbose=verbose)

    # ---- Stage 2: XGBoost fitting ----
    with Timer(device=None) as stage2_timer:   # XGBoost is CPU-bound
        model.fit_xgb(X_tr, y_tr)

    train_sec_total = stage1_timer.elapsed + stage2_timer.elapsed
    peak_mem_mb = get_gpu_peak_memory_mb(device)

    # ---- Inference timing ----
    X_te_dev = X_te.to(device)
    inf_stats = measure_inference_time(model.predict, X_te_dev, device,
                                       n_warmup=5, n_runs=20)

    # ---- Test accuracy ----
    y_true = y_te.numpy()
    y_pred = model.predict(X_te)
    acc = accuracy_score(y_true, y_pred)
    p, r, f, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(8)))

    if verbose:
        print(f"  TEST: acc={acc:.4f}  P={p:.4f}  R={r:.4f}  F1={f:.4f}")
        print(f"  COMPUTE: stage1(LSTM)={stage1_timer.elapsed:.1f}s  "
              f"stage2(XGB)={stage2_timer.elapsed:.1f}s  "
              f"total={train_sec_total:.1f}s  "
              f"inf={inf_stats['per_sample_ms']:.3f}ms/sample")

    return {
        "scenario": scenario_idx, "model": "LSTM_XGB",
        "n_train": len(X_tr),
        "accuracy": acc, "precision": p, "recall": r, "f1": f,
        "confusion": cm,
        "n_params": n_params,
        "train_sec": round(train_sec_total, 2),
        "train_sec_stage1": round(stage1_timer.elapsed, 2),
        "train_sec_stage2": round(stage2_timer.elapsed, 2),
        "inf_ms_per_sample": round(inf_stats["per_sample_ms"], 4),
        "peak_mem_mb": round(peak_mem_mb, 1),
    }


## Multi-Seed Evaluation (LSTM-XGB)

Repeats LSTM-XGB training across model-training seeds 0-4, holding the data
split fixed (`DATA_SPLIT_SEED = 20260827`, unchanged across seeds and
stages). This is a **repeated-restart / seed-variance study**, not Monte
Carlo cross-validation -- the train/val/test partition itself never
changes, only the model's own stochastic initialization/shuffling
(LSTM weight init, minibatch order, XGBoost's `random_state`). That's
deliberate: it isolates model-training variance from data-sampling
variance, unlike Li et al.'s protocol, where resampling across runs
conflates the two. It also means results are paired across seeds and
across stages, since every seed sees the exact same held-out rows.

The single seed=0 run above is kept as-is for continuity with prior
discussion; this section adds the full 5-seed picture around it.


In [5]:
from multiseed import run_multi_seed, aggregate_results, summary_table, per_class_accuracy, DEFAULT_SEEDS

print(f"LSTM-XGB across {len(DEFAULT_SEEDS)} seeds: {DEFAULT_SEEDS}")
lstm_xgb_seed_results = run_multi_seed(
    run_scenario, seeds=DEFAULT_SEEDS, scenario_idx=0, dct=dct, cols=cols, device=device)

lstm_xgb_agg = aggregate_results(lstm_xgb_seed_results)
print("\nLSTM-XGB, mean +/- std across seeds:")
for m, s in lstm_xgb_agg["stats"].items():
    print(f"  {m:10s}  {s['mean']:.4f} +/- {s['std']:.4f}   (min={s['min']:.4f} max={s['max']:.4f})")


LSTM-XGB across 5 seeds: [0, 1, 2, 3, 4]
  seed=0  acc=0.9938  P=0.9939  R=0.9938  F1=0.9938  train_sec=40.3
  seed=1  acc=0.8850  P=0.9016  R=0.8850  F1=0.8835  train_sec=37.5
  seed=2  acc=0.9883  P=0.9889  R=0.9883  F1=0.9882  train_sec=37.9
  seed=3  acc=0.9962  P=0.9964  R=0.9962  F1=0.9962  train_sec=38.0
  seed=4  acc=0.9888  P=0.9892  R=0.9888  F1=0.9887  train_sec=38.3

LSTM-XGB, mean +/- std across seeds:
  accuracy    0.9704 +/- 0.0479   (min=0.8850 max=0.9962)
  precision   0.9740 +/- 0.0406   (min=0.9016 max=0.9964)
  recall      0.9704 +/- 0.0479   (min=0.8850 max=0.9962)
  f1          0.9701 +/- 0.0485   (min=0.8835 max=0.9962)


In [6]:
# CNN-LSTM training pipeline, ported from cnn-lstm.ipynb onto the same
# dct-based data flow used above for LSTM_XGB. to_tensors() and
# load_scenario() are reused as-is from the previous cell -- CNN_LSTM is a
# plain end-to-end classifier, though, so instead of LSTM_XGB's two-stage
# fit it needs a standard epoch-by-epoch train/val loop with DataLoaders.

def make_model(model_cls, device, seed=0, **kwargs):
    """Fresh CNN-LSTM with Xavier init for Conv/Linear; default PyTorch init
    for LSTM (MATLAB's Glorot applies to Conv and FC, LSTM uses its own)."""
    torch.manual_seed(seed)
    model = model_cls(n_classes=8, **kwargs).to(device)
    for m in model.modules():
        if isinstance(m, nn.Conv1d):
            nn.init.xavier_uniform_(m.weight)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight)
            nn.init.zeros_(m.bias)
    return model


def make_loaders(X_tr, y_tr, X_va, y_va, X_te, y_te, batch_size=50, seed=0):
    g = torch.Generator().manual_seed(seed)
    perm = torch.randperm(len(X_tr), generator=g)
    X_tr_s, y_tr_s = X_tr[perm], y_tr[perm]
    train_loader = DataLoader(TensorDataset(X_tr_s, y_tr_s),
                              batch_size=batch_size, shuffle=False)
    val_loader   = DataLoader(TensorDataset(X_va, y_va), batch_size=batch_size)
    test_loader  = DataLoader(TensorDataset(X_te, y_te), batch_size=batch_size)
    return train_loader, val_loader, test_loader


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = total_correct = total_n = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss    += loss.item() * xb.size(0)
        total_correct += (logits.argmax(1) == yb).sum().item()
        total_n       += xb.size(0)
    return total_loss / total_n, total_correct / total_n


@torch.no_grad()
def evaluate_loader(model, loader, criterion, device):
    model.eval()
    total_loss = total_correct = total_n = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        total_loss    += criterion(logits, yb).item() * xb.size(0)
        total_correct += (logits.argmax(1) == yb).sum().item()
        total_n       += xb.size(0)
    return total_loss / total_n, total_correct / total_n


@torch.no_grad()
def predict_all(model, loader, device):
    model.eval()
    y_true, y_pred = [], []
    for xb, yb in loader:
        logits = model(xb.to(device))
        y_pred.append(logits.argmax(1).cpu().numpy())
        y_true.append(yb.numpy())
    return np.concatenate(y_true), np.concatenate(y_pred)


def run_scenario_cnn_lstm(scenario_idx, dct, cols, model_cls, device,
                          epochs=70, lr=1e-2, weight_decay=1e-4, seed=0, verbose=True):
    (X_tr, y_tr), (X_va, y_va), (X_te, y_te) = load_scenario(dct, cols)
    train_loader, val_loader, test_loader = make_loaders(
        X_tr, y_tr, X_va, y_va, X_te, y_te, batch_size=50, seed=seed)

    model = make_model(model_cls, device, seed=seed)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr,
                           betas=(0.9, 0.999), eps=1e-8,
                           weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)

    n_params, params_by_type = count_parameters(model)
    reset_gpu_peak_memory(device)

    if verbose:
        print(f"\n=== Scenario {scenario_idx} ({model_cls.__name__}) ===")
        print(f"  params: {n_params:,}  breakdown: {params_by_type}")

    with Timer(device) as train_timer:
        for ep in range(1, epochs + 1):
            tr_loss, tr_acc = train_one_epoch(model, train_loader,
                                              criterion, optimizer, device)
            va_loss, va_acc = evaluate_loader(model, val_loader,
                                              criterion, device)
            scheduler.step()
            if verbose and (ep == 1 or ep % 10 == 0 or ep == epochs):
                print(f"  ep {ep:3d} | train loss {tr_loss:.4f} acc {tr_acc:.3f}"
                      f" | val loss {va_loss:.4f} acc {va_acc:.3f}")

    peak_mem_mb = get_gpu_peak_memory_mb(device)

    # ---- Inference timing ----
    X_te_dev = X_te.to(device)
    def _predict(X):
        model.eval()
        with torch.no_grad():
            return model(X).argmax(1)
    inf_stats = measure_inference_time(_predict, X_te_dev, device,
                                       n_warmup=5, n_runs=20)

    # ---- Test accuracy ----
    y_true, y_pred = predict_all(model, test_loader, device)
    acc = accuracy_score(y_true, y_pred)
    p, r, f, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(8)))

    if verbose:
        print(f"  TEST: acc={acc:.4f}  P={p:.4f}  R={r:.4f}  F1={f:.4f}")
        print(f"  COMPUTE: train={train_timer.elapsed:.1f}s  "
              f"inf={inf_stats['per_sample_ms']:.3f}ms/sample  "
              f"peak_mem={peak_mem_mb:.1f}MB")

    return {
        "scenario":    scenario_idx,
        "model":       model_cls.__name__,
        "n_train":     len(X_tr),
        "accuracy":    acc,
        "precision":   p,
        "recall":      r,
        "f1":          f,
        "confusion":   cm,
        "n_params":    n_params,
        "train_sec":   round(train_timer.elapsed, 2),
        "inf_ms_per_sample": round(inf_stats["per_sample_ms"], 4),
        "peak_mem_mb": round(peak_mem_mb, 1),
    }


## Multi-Seed Evaluation (CNN-LSTM-v2)

Same 5 seeds, same fixed data split. The single seed=0 run above is kept
for continuity; this adds the full picture.


In [ ]:
MODEL_CLS = CNN_LSTM_v2  # or CNN_LSTM_v1

print(f"CNN-LSTM-v2 across {len(DEFAULT_SEEDS)} seeds: {DEFAULT_SEEDS}")
cnn_lstm_seed_results = run_multi_seed(
    run_scenario_cnn_lstm, seeds=DEFAULT_SEEDS, scenario_idx=1, dct=dct, cols=cols,
    model_cls=MODEL_CLS, device=device)

cnn_lstm_agg = aggregate_results(cnn_lstm_seed_results)
print("\nCNN-LSTM-v2, mean +/- std across seeds:")
for m, s in cnn_lstm_agg["stats"].items():
    print(f"  {m:10s}  {s['mean']:.4f} +/- {s['std']:.4f}   (min={s['min']:.4f} max={s['max']:.4f})")


CNN-LSTM-v2 across 5 seeds: [0, 1, 2, 3, 4]
  seed=0  acc=1.0000  P=1.0000  R=1.0000  F1=1.0000  train_sec=63.6
  seed=1  acc=1.0000  P=1.0000  R=1.0000  F1=1.0000  train_sec=63.0
  seed=2  acc=1.0000  P=1.0000  R=1.0000  F1=1.0000  train_sec=78.7
  seed=3  acc=1.0000  P=1.0000  R=1.0000  F1=1.0000  train_sec=73.4
  seed=4  acc=0.9996  P=0.9996  R=0.9996  F1=0.9996  train_sec=70.1

CNN-LSTM-v2, mean +/- std across seeds:
  accuracy    0.9999 +/- 0.0002   (min=0.9996 max=1.0000)
  precision   0.9999 +/- 0.0002   (min=0.9996 max=1.0000)
  recall      0.9999 +/- 0.0002   (min=0.9996 max=1.0000)
  f1          0.9999 +/- 0.0002   (min=0.9996 max=1.0000)


## Combined Multi-Seed Summary

Mean +/- std across the 5 seeds for both models, plus per-class accuracy
averaged across seeds (diagonal of the seed-averaged, row-normalized
confusion matrix). Results are pickled for the eventual Stage-1-vs-Stage-2
(and later Stage-3/4) paired comparison via `multiseed.paired_diff`.


In [9]:
combined = summary_table({"LSTM-XGB": lstm_xgb_agg, "CNN-LSTM-v2": cnn_lstm_agg})
print(combined.round(4).to_string(index=False))

class_names = ["Normal", "F1", "F2", "F3", "F4", "F5", "F6", "F7"]
pc = pd.DataFrame({
    "LSTM-XGB": per_class_accuracy(lstm_xgb_agg["mean_confusion_rate"], class_names),
    "CNN-LSTM-v2": per_class_accuracy(cnn_lstm_agg["mean_confusion_rate"], class_names),
})
print("\nPer-class accuracy (mean confusion diagonal across seeds):")
print(pc.round(4).to_string())

import pickle
with open("stage1_seed_results.pkl", "wb") as f:
    pickle.dump({"lstm_xgb": lstm_xgb_agg, "cnn_lstm_v2": cnn_lstm_agg}, f)
print("\nSaved stage1_seed_results.pkl")


      model  n_seeds  accuracy_mean  accuracy_std  precision_mean  precision_std  recall_mean  recall_std  f1_mean  f1_std
   LSTM-XGB        5         0.9704        0.0479          0.9740         0.0406       0.9704      0.0479   0.9701  0.0485
CNN-LSTM-v2        5         0.9999        0.0002          0.9999         0.0002       0.9999      0.0002   0.9999  0.0002

Per-class accuracy (mean confusion diagonal across seeds):
        LSTM-XGB  CNN-LSTM-v2
Normal    0.9147       1.0000
F1        0.9487       1.0000
F2        0.9953       1.0000
F3        0.9793       0.9993
F4        0.9993       1.0000
F5        0.9980       1.0000
F6        0.9293       1.0000
F7        0.9987       1.0000

Saved stage1_seed_results.pkl
